# qlib_recorder_adapter_v1

目标：验证 `src/analysis` 可以读取 qlib recorder/analyzer 常见产物，把 portfolio report、risk analysis、positions/orders 统一成 AnalysisReport sidecar。

In [1]:
from pathlib import Path
import json
import os
import sys

import pandas as pd

REPO = Path('/Users/rziiiii/dev/workspace')
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from src.analysis import build_qlib_recorder_analysis_report, load_qlib_recorder_analysis, write_analysis_report
from src.core import DatasetSnapshot

CARD = Path('research/qlib_recorder_adapter_v1')
RECORDER_ARTIFACTS = CARD / 'qlib_artifacts'
REPORT_DIR = CARD / 'analysis_report'
CARD.mkdir(parents=True, exist_ok=True)
RECORDER_ARTIFACTS.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)
{
    'card': str(CARD),
    'recorder_artifacts': str(RECORDER_ARTIFACTS),
    'report_dir': str(REPORT_DIR),
}


{'card': 'research/qlib_recorder_adapter_v1',
 'recorder_artifacts': 'research/qlib_recorder_adapter_v1/qlib_artifacts',
 'report_dir': 'research/qlib_recorder_adapter_v1/analysis_report'}

## 构造 qlib-like recorder artifacts

这里使用本地 pkl/csv 模拟 qlib recorder 保存的 `report_normal_1day`、`risk_analysis_1day`、positions 和 orders。生产接入真实 recorder 时走同一个 `load_qlib_recorder_analysis` 接口。

In [2]:
dates = pd.date_range('2026-01-01', periods=6, freq='D')
portfolio_report = pd.DataFrame(
    {
        'return': [0.0, 0.018, -0.012, 0.024, -0.008, 0.031],
        'bench': [0.0, 0.011, -0.015, 0.015, -0.006, 0.020],
        'cost': [0.0, 0.001, 0.001, 0.002, 0.001, 0.002],
    },
    index=dates,
)
risk_analysis = pd.DataFrame(
    {
        'metric': ['annualized_return', 'information_ratio', 'max_drawdown'],
        'value': [0.58, 1.41, -0.067],
    }
)
positions = pd.DataFrame(
    {
        'datetime': dates[:3],
        'instrument': ['SOXL', 'SOXX', 'CASH'],
        'weight': [0.72, 0.18, 0.10],
    }
)
orders = pd.DataFrame(
    {
        'datetime': dates[1:4],
        'instrument': ['SOXL', 'SOXL', 'SOXX'],
        'amount': [120, -30, 40],
        'deal_price': [35.2, 34.8, 248.5],
    }
)

portfolio_report.to_pickle(RECORDER_ARTIFACTS / 'report_normal_1day.pkl')
risk_analysis.to_csv(RECORDER_ARTIFACTS / 'risk_analysis_1day.csv', index=False)
positions.to_csv(RECORDER_ARTIFACTS / 'positions_normal_1day.csv', index=False)
orders.to_csv(RECORDER_ARTIFACTS / 'orders_normal_1day.csv', index=False)
sorted(path.name for path in RECORDER_ARTIFACTS.iterdir())


['orders_normal_1day.csv',
 'positions_normal_1day.csv',
 'report_normal_1day.pkl',
 'risk_analysis_1day.csv']

## 读取并标准化

adapter 负责把 qlib artifact 名称和表结构转成项目内稳定命名，核心输出包括 `qlib_portfolio_report` 和 `qlib_risk_analysis`。

In [3]:
recorder_analysis = load_qlib_recorder_analysis(RECORDER_ARTIFACTS, model_name='qlib_recorder_demo')
recorder_analysis.summary


{'schema_version': 'analysis.qlib_recorder.v1',
 'source_type': 'artifact_dir',
 'model': 'qlib_recorder_demo',
 'loaded_artifacts': {'qlib_portfolio_report': 'report_normal_1day.pkl',
  'qlib_risk_analysis': 'risk_analysis_1day.csv',
  'qlib_positions': 'positions_normal_1day.csv',
  'qlib_orders': 'orders_normal_1day.csv'},
 'missing_artifact_keys': [],
 'missing_optional_artifact_keys': ['qlib_indicator_analysis'],
 'table_counts': {'qlib_orders': 3,
  'qlib_portfolio_report': 6,
  'qlib_positions': 3,
  'qlib_risk_analysis': 3},
 'notes': 'Qlib recorder artifacts are normalized into stable src/analysis tables before dashboard/notebook consumption.'}

In [4]:
portfolio = recorder_analysis.tables['qlib_portfolio_report']
risk = recorder_analysis.tables['qlib_risk_analysis']
positions_table = recorder_analysis.tables['qlib_positions']
orders_table = recorder_analysis.tables['qlib_orders']

checks = {
    'portfolio_columns': portfolio.columns.tolist(),
    'risk_metrics': risk['metric'].tolist(),
    'table_counts': recorder_analysis.summary['table_counts'],
    'missing_required': recorder_analysis.summary['missing_artifact_keys'],
    'missing_optional': recorder_analysis.summary['missing_optional_artifact_keys'],
    'final_nav': float(portfolio['nav'].iloc[-1]),
}
assert not checks['missing_required']
assert {'strategy_return', 'benchmark_return', 'excess_return', 'nav'}.issubset(portfolio.columns)
assert {'annualized_return', 'information_ratio', 'max_drawdown'} == set(risk['metric'])
checks


{'portfolio_columns': ['date',
  'model',
  'strategy_return',
  'benchmark_return',
  'excess_return',
  'nav',
  'benchmark_nav',
  'cost',
  'source_artifact'],
 'risk_metrics': ['annualized_return', 'information_ratio', 'max_drawdown'],
 'table_counts': {'qlib_orders': 3,
  'qlib_portfolio_report': 6,
  'qlib_positions': 3,
  'qlib_risk_analysis': 3},
 'missing_required': [],
 'missing_optional': ['qlib_indicator_analysis'],
 'final_nav': 1.053355619909632}

In [5]:
portfolio

,date,model,strategy_return,benchmark_return,excess_return,nav,benchmark_nav,cost,source_artifact
0,2026-01-01,qlib_recorder_demo,0.000,0.000,0.000,1.000000,1.000000,0.000,report_normal_1day.pkl
1,2026-01-02,qlib_recorder_demo,0.018,0.011,0.007,1.018000,1.011000,0.001,report_normal_1day.pkl
2,2026-01-03,qlib_recorder_demo,-0.012,-0.015,0.003,1.005784,0.995835,0.001,report_normal_1day.pkl
3,2026-01-04,qlib_recorder_demo,0.024,0.015,0.009,1.029923,1.010773,0.002,report_normal_1day.pkl
4,2026-01-05,qlib_recorder_demo,-0.008,-0.006,-0.002,1.021683,1.004708,0.001,report_normal_1day.pkl
5,2026-01-06,qlib_recorder_demo,0.031,0.020,0.011,1.053356,1.024802,0.002,report_normal_1day.pkl


In [6]:
risk

,model,metric,value,return_kind,source_artifact
0,qlib_recorder_demo,annualized_return,0.580,qlib_risk,risk_analysis_1day.csv
1,qlib_recorder_demo,information_ratio,1.410,qlib_risk,risk_analysis_1day.csv
2,qlib_recorder_demo,max_drawdown,-0.067,qlib_risk,risk_analysis_1day.csv


## 写出 AnalysisReport

用统一报告层写出 dashboard/notebook/promotion 可读取的 sidecar 和 manifest。

In [7]:
report = build_qlib_recorder_analysis_report(
    artifact_id='qlib_recorder_adapter_v1',
    recorder_analysis=recorder_analysis,
    summary={
        'status': 'qlib_recorder_adapter_completed',
        'note': '中文 review 入口：qlib recorder 产物已统一进入 AnalysisReport。',
    },
)
dataset = DatasetSnapshot(
    name='qlib_recorder_demo',
    universe=('SOXL', 'SOXX'),
    start='2026-01-01',
    end='2026-01-06',
    calendar_path='data/qlib/soxl/calendars/day.txt',
    data_paths=(str(RECORDER_ARTIFACTS),),
)
export_result = write_analysis_report(report, REPORT_DIR, dataset=dataset)
summary = json.loads((REPORT_DIR / 'analysis_summary.json').read_text(encoding='utf-8'))
manifest = json.loads((REPORT_DIR / 'analysis_manifest.json').read_text(encoding='utf-8'))
assert summary['schema_version'] == 'analysis.report.v1'
assert manifest['metadata']['source_type'] == 'artifact_dir'
assert manifest['paths']['qlib_portfolio_report'] == 'qlib_portfolio_report.csv'
assert len(manifest['manifest_hash']) == 64
{
    'export_result': export_result,
    'summary_schema': summary['schema_version'],
    'manifest_hash_len': len(manifest['manifest_hash']),
    'paths': manifest['paths'],
}


{'export_result': {'schema_version': 'analysis.report.export.v1',
  'artifact_id': 'qlib_recorder_adapter_v1',
  'artifact_type': 'analysis_report',
  'paths': {'summary': 'analysis_summary.json',
   'qlib_portfolio_report': 'qlib_portfolio_report.csv',
   'qlib_risk_analysis': 'qlib_risk_analysis.csv',
   'qlib_positions': 'qlib_positions.csv',
   'qlib_orders': 'qlib_orders.csv',
   'manifest': 'analysis_manifest.json'},
  'table_counts': {'qlib_orders': 3,
   'qlib_portfolio_report': 6,
   'qlib_positions': 3,
   'qlib_risk_analysis': 3},
  'manifest': 'analysis_manifest.json'},
 'summary_schema': 'analysis.report.v1',
 'manifest_hash_len': 64,
 'paths': {'qlib_orders': 'qlib_orders.csv',
  'qlib_portfolio_report': 'qlib_portfolio_report.csv',
  'qlib_positions': 'qlib_positions.csv',
  'qlib_risk_analysis': 'qlib_risk_analysis.csv',
  'summary': 'analysis_summary.json'}}

## 写出本实验摘要

card 根目录保留本实验摘要；完整标准报告保留在 `analysis_report/`。

In [8]:
run_summary = {
    'schema_version': 'research.qlib_recorder_adapter_v1',
    'status': 'completed',
    'recorder_artifacts': str(RECORDER_ARTIFACTS),
    'analysis_report': str(REPORT_DIR),
    'loaded_artifacts': recorder_analysis.summary['loaded_artifacts'],
    'missing_required': recorder_analysis.summary['missing_artifact_keys'],
    'missing_optional': recorder_analysis.summary['missing_optional_artifact_keys'],
    'table_counts': recorder_analysis.summary['table_counts'],
    'analysis_manifest_hash': manifest['manifest_hash'],
    'conclusion': 'src/analysis 已支持从 qlib recorder-like artifact 目录读取 portfolio/risk/positions/orders 并写成统一 AnalysisReport。',
}
(CARD / 'analysis_summary.json').write_text(json.dumps(run_summary, ensure_ascii=False, indent=2), encoding='utf-8')
run_summary


{'schema_version': 'research.qlib_recorder_adapter_v1',
 'status': 'completed',
 'recorder_artifacts': 'research/qlib_recorder_adapter_v1/qlib_artifacts',
 'analysis_report': 'research/qlib_recorder_adapter_v1/analysis_report',
 'loaded_artifacts': {'qlib_portfolio_report': 'report_normal_1day.pkl',
  'qlib_risk_analysis': 'risk_analysis_1day.csv',
  'qlib_positions': 'positions_normal_1day.csv',
  'qlib_orders': 'orders_normal_1day.csv'},
 'missing_required': [],
 'missing_optional': ['qlib_indicator_analysis'],
 'table_counts': {'qlib_orders': 3,
  'qlib_portfolio_report': 6,
  'qlib_positions': 3,
  'qlib_risk_analysis': 3},
 'analysis_manifest_hash': '47081a14de710374abeba21bba3d0368f3bdbf2fdf248c27be8bf097e9b84a43',
 'conclusion': 'src/analysis 已支持从 qlib recorder-like artifact 目录读取 portfolio/risk/positions/orders 并写成统一 AnalysisReport。'}

## 中文结论

- 做了什么：新增并验证 `src/analysis` 的 qlib recorder adapter，支持从 recorder-like object 或本地 artifact 目录读取 qlib 常见 portfolio/risk/positions/orders 产物。
- 核心指标：本轮样例读取 4 类 artifact，生成 `qlib_portfolio_report.csv`、`qlib_risk_analysis.csv`、`qlib_positions.csv`、`qlib_orders.csv` 和 `analysis_manifest.json`；manifest hash 为 64 位。
- regime 差异：本轮重点是 qlib recorder/analyzer 读取层，还没有重新切 regime；后续这些 qlib 表可以和现有 regime/window 分析报告在同一 `AnalysisReport` 契约下组合。
- 是否晋升：这是框架能力，不是策略晋升；可以作为未来真实 qlib recorder、dashboard 和 production notebook 的稳定输入层。
- 风险：样例使用本地 qlib-like artifact，不依赖真实 qlib experiment server；真实 recorder 接入仍需在生产 notebook 里验证 artifact 命名差异。
- 下一步：让 dashboard export 优先读取 `analysis_manifest.json`，并把真实 qlib recorder 的 portfolio/risk artifact 接入同一 adapter。